# MLP with Dimensionless Parameters — Case 3
Cone calorimeter (helical coil) → differential area element  
Features: h/d, k/d  |  Fixed constants: d=30 mm, p=0.1 mm, R=40 mm, H=65 mm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import Normalize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import time

size_text = 24
plt.rcParams.update({
    'font.family': 'serif', 'text.usetex': False,
    'axes.titlesize': size_text, 'axes.labelsize': size_text,
    'xtick.labelsize': size_text, 'ytick.labelsize': size_text,
    'legend.fontsize': size_text,
})
w, h_fig = 8, 6


## Dataset and dimensionless features

In [ ]:
# Fixed geometric constants (from ISO 5660-1 cone calorimeter)
d_ref = 30.0   # vertical distance, mm  — reference length
p_c   =  0.1   # element diameter, mm  — fixed in this dataset
R_c   = 40.0   # coil radius (R1=R2=40), mm
H_c   = 65.0   # coil height, mm

df = pd.read_csv("../data/Case_3.csv")

df['h_d'] = df['h'] / d_ref   # lateral x / distance
df['k_d'] = df['k'] / d_ref   # lateral y / distance

feat_names = ['h_d', 'k_d']
X = df[feat_names].values
y = df['VF'].values.reshape(-1, 1)

print(f"Dataset shape: {df.shape}")
print(f"\nDimensionless feature ranges:")
for f in feat_names:
    print(f"  {f:>5s}: [{df[f].min():.4f}, {df[f].max():.4f}]  ({df[f].nunique()} unique)")
print(f"\nFixed constants (zero variance in this dataset — not used as features):")
print(f"  p/d = {p_c/d_ref:.5f}   R/d = {R_c/d_ref:.4f}   H/d = {H_c/d_ref:.4f}")
print(f"\nNote: p, d, R, H are all fixed in this dataset.")
print(f"Adding p/d, R/d, H/d as features is correct for a future dataset")
print(f"where these parameters are also varied.")


## Train – Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}   Test: {X_test.shape[0]}")


## Scaling and tensors

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train).ravel()
y_test_scaled  = scaler_y.transform(y_test).ravel()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train_scaled.reshape(-1,1), dtype=torch.float32).to(device)
X_test_tensor  = torch.tensor(X_test_scaled,  dtype=torch.float32).to(device)
y_test_tensor  = torch.tensor(y_test_scaled.reshape(-1,1),  dtype=torch.float32).to(device)

train_dl = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=256, shuffle=True)


## MLP architecture  (2 dimensionless inputs)

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64), nn.GELU(),
            nn.Linear(64, 64),     nn.GELU(),
            nn.Linear(64, 32),     nn.GELU(),
            nn.Linear(32, 1)
        )
    def forward(self, x): return self.net(x)

model = MLP().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  ({n_params*4/1024:.1f} KB)")


## Training

In [ ]:
loss_fn   = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 300
train_losses, val_losses = [], []

t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        tl = loss_fn(model(X_train_tensor), y_train_tensor).item()
        vl = loss_fn(model(X_test_tensor),  y_test_tensor).item()
    train_losses.append(tl); val_losses.append(vl)

    if epoch % 50 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | Train MSE: {tl:.4e} | Test MSE: {vl:.4e}")

T_train = time.perf_counter() - t0
model.eval()
if torch.cuda.is_available(): torch.cuda.synchronize()
t2 = time.perf_counter()
_ = model(X_test_tensor).detach()
if torch.cuda.is_available(): torch.cuda.synchronize()
T_pred = (time.perf_counter() - t2) / X_test_tensor.shape[0]
print(f"\nTraining time : {T_train:.2f} s")
print(f"Inference time: {T_pred*1e6:.4f} µs/sample")


## Learning curves

In [ ]:
fig, ax = plt.subplots(figsize=(w, h_fig), dpi=150)
ax.plot(train_losses, 'r-', linewidth=2, label='Train MSE')
ax.plot(val_losses,   'b-', linewidth=2, label='Test MSE')
ax.set_title("Learning Curves — MLP dimensionless (Case 3)")
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss (scaled)")
ax.legend(); ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
plt.tight_layout()
plt.savefig("learning_curve_mlp_dimless_C3.pdf", dpi=400, bbox_inches='tight')
plt.show()


## Metrics

In [ ]:
model.eval()
with torch.no_grad():
    ytr_pred_s = model(X_train_tensor).cpu().numpy()
    yte_pred_s = model(X_test_tensor).cpu().numpy()

y_train_pred = scaler_y.inverse_transform(ytr_pred_s)
y_test_pred  = scaler_y.inverse_transform(yte_pred_s)
y_train_true = scaler_y.inverse_transform(y_train_tensor.cpu().numpy())
y_test_true  = scaler_y.inverse_transform(y_test_tensor.cpu().numpy())

def metrics(yt, yp, label):
    mse  = mean_squared_error(yt, yp)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    mape = np.mean(np.abs((yt - yp) / yt)) * 100
    print(f"{label}:  MSE={mse:.4e}  RMSE={rmse:.4e}  MAE={mae:.4e}  R²={r2:.6f}  MAPE={mape:.4f}%")
    return mse, rmse, mae, r2, mape

metrics(y_train_true, y_train_pred, "Train")
_, _, _, _, mape_test = metrics(y_test_true, y_test_pred, "Test ")


## Prediction error bands

In [ ]:
rel_err = np.abs((y_test_true - y_test_pred) / y_test_true)
print("Percentage of test predictions within tolerance:")
for tol in [0.01, 0.02, 0.05, 0.10]:
    print(f"  < {tol*100:.0f}%  →  {np.mean(rel_err < tol)*100:.2f}%")


## Comparison: dimensional  vs  dimensionless

In [ ]:
# Original dimensional model results (from Neural_network_case_3.ipynb)
mape_dim = 0.2741
r2_dim   = 0.996621
rmse_dim = 2.813e-3

print(f"{'Metric':<20} {'Dimensional (4 feat)':>22} {'Dimensionless (2 feat)':>22}")
print("-" * 66)
print(f"{'MAPE test (%)':<20} {mape_dim:>22.4f} {mape_test:>22.4f}")
print(f"{'R² test':<20} {r2_dim:>22.6f} {r2_score(y_test_true, y_test_pred):>22.6f}")
print(f"{'RMSE test':<20} {rmse_dim:>22.4e} {np.sqrt(mean_squared_error(y_test_true, y_test_pred)):>22.4e}")
print()
print("Note: the original 4-feature model (h, k, p, d) is effectively")
print("a 2-feature model since p=0.1 and d=30 are constant in this dataset.")


## Predicted vs actual  (test set)

In [ ]:
fig, ax = plt.subplots(figsize=(w, h_fig), dpi=150)
shadow = [pe.SimpleLineShadow(offset=(1,-1), alpha=0.25), pe.Normal()]
sc = ax.scatter(y_test_true, y_test_pred, s=8, alpha=0.7,
                color='royalblue', edgecolor='black', linewidth=0.1,
                label='Predictions')
sc.set_path_effects(shadow)
mn = float(min(y_test_true.min(), y_test_pred.min()))
mx = float(max(y_test_true.max(), y_test_pred.max()))
ax.plot([mn, mx], [mn, mx], 'k--', linewidth=1.5, label='Ideal fit')
ax.set_title("Real vs Predicted — MLP dimensionless (Case 3)")
ax.set_xlabel("True values"); ax.set_ylabel("Predicted values")
ax.legend(); ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
plt.tight_layout()
plt.savefig("real_vs_pred_mlp_dimless_C3.pdf", dpi=400, bbox_inches='tight')
plt.show()


## Prediction heatmap  (h/d  vs  k/d)

In [ ]:
hd_vals = np.linspace(df['h_d'].min(), df['h_d'].max(), 300)
kd_vals = np.linspace(df['k_d'].min(), df['k_d'].max(), 300)
Hd, Kd  = np.meshgrid(hd_vals, kd_vals)

X_grid   = np.stack([Hd.ravel(), Kd.ravel()], axis=1)
X_grid_s = scaler_X.transform(X_grid)

model.eval()
with torch.no_grad():
    F_s = model(torch.tensor(X_grid_s, dtype=torch.float32).to(device)).cpu().numpy()

VF_grid = scaler_y.inverse_transform(F_s).reshape(Hd.shape)

norm = Normalize(vmin=np.percentile(VF_grid, 10), vmax=np.percentile(VF_grid, 90))
fig, ax = plt.subplots(figsize=(w, h_fig), dpi=150)
img = ax.imshow(VF_grid, extent=(hd_vals.min(), hd_vals.max(), kd_vals.min(), kd_vals.max()),
                origin='lower', aspect='auto', cmap='magma', norm=norm)
cbar = fig.colorbar(img, ax=ax)
cbar.set_label(r'$F_{dA	o c}$ (–)')
ax.set_title(f"MLP (dimensionless) — d = {d_ref:.0f} mm")
ax.set_xlabel("h / d"); ax.set_ylabel("k / d")
plt.tight_layout()
plt.savefig("heatmap_mlp_dimless_C3.pdf", dpi=400, bbox_inches='tight')
plt.show()


# SHAP Analysis

## Setup and compute SHAP values

In [ ]:
import shap

model.to(device); model.eval()
torch.manual_seed(42)
bg  = X_train_tensor[torch.randperm(X_train_tensor.shape[0])[:200]]
expl = shap.GradientExplainer(model, bg)

n_ex = min(1000, X_test_tensor.shape[0])
sv   = expl.shap_values(X_test_tensor[:n_ex])
sv   = sv[0] if isinstance(sv, list) else sv
if sv.ndim == 3: sv = sv.squeeze(-1)

feat_shap = ['h/d', 'k/d']
X_ex_orig = X_test[:n_ex]

print(f"SHAP shape: {sv.shape}")
print("Mean |SHAP| per feature:")
for i, nm in enumerate(feat_shap):
    print(f"  {nm}: {np.abs(sv[:,i]).mean():.6f}")


## Bar plot

In [ ]:
plt.figure(figsize=(6, 4), dpi=150)
shap.summary_plot(sv, X_ex_orig, feature_names=feat_shap, plot_type='bar', show=False)
plt.tight_layout()
plt.savefig("shap_bar_dimless_C3.pdf", dpi=400, bbox_inches='tight')
plt.show()


## Beeswarm plot

In [ ]:
plt.figure(figsize=(7, 5), dpi=150)
shap.summary_plot(sv, X_ex_orig, feature_names=feat_shap, show=False)
plt.tight_layout()
plt.savefig("shap_beeswarm_dimless_C3.pdf", dpi=400, bbox_inches='tight')
plt.show()


## Dependence plots

In [ ]:
for i, nm in enumerate(feat_shap):
    plt.figure(figsize=(8, 5), dpi=150)
    shap.dependence_plot(i, sv, X_ex_orig, feature_names=feat_shap, show=False)
    plt.tight_layout()
    fname = nm.replace('/', '_')
    plt.savefig(f"shap_dependence_{fname}_dimless_C3.pdf", dpi=400, bbox_inches='tight')
    plt.show()
